In [56]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped



polars.config.Config

In [57]:
market_response_path = "/Users/oceansxyz/alphanode-metal/logs/market/market_20260618_220038.jsonl"

#### Load JSONL - Binace Market WS

In [58]:
#Raw binance reponse for market web socket - pretty much a static
book_ticker_schema = pl.Struct([
    pl.Field("ts", pl.Int64),
    pl.Field("frame", pl.Struct([
        pl.Field("u", pl.Int64),
        pl.Field("s", pl.String),
        pl.Field("b", pl.String),
        pl.Field("B", pl.String),
        pl.Field("a", pl.String),
        pl.Field("A", pl.String),
    ])),
])

with open(market_response_path) as f:
    metadata_json = f.readline().strip()
    print("Metadata Raw Line:", metadata_json)
    df_market = (
        pl.read_csv(
            f,
            has_header=False,
            new_columns=["json_str"],
            separator="\n",
            infer_schema=False,
        )
        .select(pl.col("json_str").str.json_decode(dtype=book_ticker_schema))
        .unnest("json_str")
        .unnest("frame")
        .with_columns([
            pl.col("u").cast(pl.Int64),
            pl.col("s"),
            pl.col("b").cast(pl.Float64),
            pl.col("B").cast(pl.Float64),
            pl.col("a").cast(pl.Float64),
            pl.col("A").cast(pl.Float64),
        ])
    )


# Select relevant cols for alpha calulation with identifiable file names
market_rates = df_market.select(
    col("ts"), col("s").alias("symbol"), col("b").alias("bid"), col("a").alias("ask")
)

# duration of web socket listen
min_ts = market_rates.min().select("ts").item()
max_ts = market_rates.max().select("ts").item()
duration_m =(max_ts-min_ts)/60000


print(f"Duration of listen : {duration_m} mins")
print(market_rates.group_by("symbol").len())

Metadata Raw Line: {"ts":1781820039723,"frame":{"result":null,"id":1}}
Duration of listen : 314.55286666666666 mins
shape: (3, 2)
┌──────────┬────────┐
│ symbol   ┆ len    │
│ ---      ┆ ---    │
│ str      ┆ u32    │
╞══════════╪════════╡
│ BTCEURI  ┆ 5567   │
│ BTCUSDT  ┆ 125940 │
│ EURIUSDT ┆ 925    │
└──────────┴────────┘


In [59]:
market_rates.head()

ts,symbol,bid,ask
i64,str,f64,f64
1781820039773,"""BTCUSDT""",62957.99,62958.0
1781820040495,"""BTCEURI""",54871.89,54891.46
1781820040615,"""BTCUSDT""",62957.99,62958.0
1781820040645,"""BTCUSDT""",62957.99,62958.0
1781820040767,"""BTCUSDT""",62957.99,62958.0


In [60]:
## Split dataframe symbolwise symbol is an orderbook
symbol_0 = market_rates.filter(col("symbol")=="BTCUSDT").select(["ts", col("bid").alias("BTCUSDT_bid"), col("ask").alias("BTCUSDT_ask")])
symbol_1 = market_rates.filter(col("symbol")=="BTCEURI").select(["ts", col("bid").alias("BTCEURI_bid"), col("ask").alias("BTCEURI_ask")])
symbol_2 = market_rates.filter(col("symbol")=="EURIUSDT").select(["ts", col("bid").alias("EURIUSDT_bid"), col("ask").alias("EURIUSDT_ask")])

In [61]:
# Forward fill missing values by copying the last known valid price down 
# for all generated ticker metrics simultaneously, skipping the timestamp.
market_rates_widened = symbol_0.join(symbol_1, on="ts", how="full", coalesce=True)\
    .join(symbol_2, on="ts", how="full", coalesce=True)\
    .sort("ts")\
    .select([
        col("ts"),
        col("*").exclude("ts").fill_null(strategy="forward")
    ]).drop_nulls()

market_rates_widened.head()

ts,BTCUSDT_bid,BTCUSDT_ask,BTCEURI_bid,BTCEURI_ask,EURIUSDT_bid,EURIUSDT_ask
i64,f64,f64,f64,f64,f64,f64
1781820048799,62957.99,62958.0,54871.89,54885.34,1.1471,1.1474
1781820048849,62957.99,62958.0,54871.89,54885.34,1.1471,1.1474
1781820049340,62957.99,62958.0,54871.89,54954.63,1.1471,1.1474
1781820049340,62957.99,62958.0,54871.89,54942.77,1.1471,1.1474
1781820049645,62957.99,62958.0,54871.89,54942.77,1.1471,1.1474


#### Notation & Model

NODE_0 = USDT NODE_1 = BTC NODE_2 = EURI  

EDGE_0 = BTCUSDT  EDGE_1 = BTCEURI  EDGE_2 = EURIUSDT  

Clockwise Walk  0→1  →1→2  →2→0  

BUY  BTCUSDT  @ASK  === 0→1  
SELL BTCEURI  @BID  === 1→2  
SELL EURIUSDT @BID  === 2→0


#### Clockwise Walk  0→1  →1→2  →2→0

In [62]:
clockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_ask"), "BTCEURI_bid", "EURIUSDT_bid")\
    .unique(subset=["BTCUSDT_ask", "BTCEURI_bid", "EURIUSDT_bid"])\
    .with_columns((col("BTCEURI_bid")*col("EURIUSDT_bid")).alias("BTCEURIUSDT_bid"))\
    .with_columns((col("BTCEURIUSDT_bid")/col("BTCUSDT_ask")).abs().alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



clockwise_walk.sort("alpha", descending=True).head(20)



ts,BTCUSDT_ask,BTCEURI_bid,EURIUSDT_bid,BTCEURIUSDT_bid,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781838912620,62600.0,54730.0,1.1463,62736.999,1.002188,21.884824
1781838912738,62623.51,54730.0,1.1463,62736.999,1.001812,18.122427
1781838912619,62650.0,54730.0,1.1463,62736.999,1.001389,13.886512
1781838912619,62651.71,54730.0,1.1463,62736.999,1.001361,13.613196
1781831501492,62900.01,54862.37,1.1473,62943.597101,1.000693,6.929586
1781821058598,62868.01,54831.18,1.1473,62907.812814,1.000633,6.331171
1781821166597,62797.87,54770.0,1.1473,62837.621,1.000633,6.329992
1781838813570,62780.01,54800.0,1.1463,62817.24,1.000593,5.930232
1781838815263,62780.05,54800.0,1.1463,62817.24,1.000592,5.923856


#### AntiClockwise Walk   0←1 ←1←2  ←2←0

Remarkably profitable - SELL BTCUSDT; BUY BTCEURI; and BUY EURIUSDT

In [63]:
anticlockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_bid"), "BTCEURI_ask", "EURIUSDT_ask")\
    .unique(subset=["BTCUSDT_bid", "BTCEURI_ask", "EURIUSDT_ask"])\
    .with_columns((col("BTCEURI_ask")*col("EURIUSDT_ask")).alias("BTCEURIUSDT_ask"))\
    .with_columns((col("BTCUSDT_bid")/col("BTCEURIUSDT_ask")).alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



anticlockwise_walk.sort("alpha", descending=True).head(20)

ts,BTCUSDT_bid,BTCEURI_ask,EURIUSDT_ask,BTCEURIUSDT_ask,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781831831520,63099.99,54950.0,1.1474,63049.63,1.000799,7.987359
1781826986559,62985.11,54862.04,1.1472,62937.732288,1.000753,7.527712
1781826992562,62985.1,54862.04,1.1472,62937.732288,1.000753,7.526123
1781826992562,62985.09,54862.04,1.1472,62937.732288,1.000752,7.524534
1781826980558,62985.08,54862.04,1.1472,62937.732288,1.000752,7.522945
1781826940555,62985.07,54862.04,1.1472,62937.732288,1.000752,7.521356
1781826992563,62985.05,54862.04,1.1472,62937.732288,1.000752,7.518179
1781826937554,62985.04,54862.04,1.1472,62937.732288,1.000752,7.51659
1781826992563,62983.99,54862.04,1.1472,62937.732288,1.000735,7.349758


In [9]:
anticlockwise_walk.filter((col("ts")==1781820341584))

ts,BTCUSDT_bid,BTCEURI_ask,EURIUSDT_ask,BTCEURIUSDT_ask,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64


In [47]:
anticlockwise_walk.filter((col("ts")<1781435059000) & (col("ts")>1781435057000))\
    .plot.line(x="ts", y="BTCUSDT_bid")

alt.Chart(...)